# Assignment 5: ML Security and Abuse Pathways (Applied)

**Course:** DNSC 6330 Responsible Machine Learning  
**Deadline:** April 30, 2026, 11:59 pm ET  
**Deliverable:** Colab-runnable notebook + short written report

This notebook implements the Lecture 05 individual applied homework using the COMPAS pipeline from Assignments 3 and 4.

## Parts
- **Part A:** PGD evasion audit on LR and GBT at `epsilon in {0.25, 0.5, 1.0, 2.0}`
- **Part B:** Label-flip poisoning loop with fairness monitoring and stealth-zone analysis
- **Part C:** Membership inference with shadow models + LR regularization sweep
- **Part D:** Reflection and governance recommendations

> Rubric anchor used throughout: **what the metric says, what it misses, what action it justifies**.

In [ ]:
# Setup: Assignment 3/4 pipeline + helpers
import subprocess, sys
pkgs = ["pandas","numpy","matplotlib","seaborn","scikit-learn","scipy"]
for p in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], capture_output=True)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.special import expit
from scipy.stats import ks_2samp

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

np.random.seed(42)

# Data load and Assignment 3 filtering
URL = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
raw_data = pd.read_csv(URL)

cols = [
    "age", "c_charge_degree", "race", "age_cat", "score_text", "sex",
    "priors_count", "days_b_screening_arrest", "decile_score",
    "is_recid", "two_year_recid", "c_jail_in", "c_jail_out"
]

df = raw_data[cols].copy()
df = df[
    df["days_b_screening_arrest"].between(-30, 30) &
    (df["is_recid"] != -1) &
    (df["c_charge_degree"] != "O") &
    (df["score_text"] != "N/A")
].reset_index(drop=True)
df["two_year_recid_num"] = df["two_year_recid"].astype(int)

numeric_features = ["age", "priors_count", "days_b_screening_arrest", "decile_score"]
category_features = ["c_charge_degree", "race", "age_cat", "sex", "score_text"]
features = numeric_features + category_features
target = "two_year_recid_num"

X = df[features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), category_features),
])

def make_lr_pipeline(C=1.0):
    return Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", LogisticRegression(max_iter=1000, C=C, random_state=42)),
    ])

def make_gbt_pipeline():
    return Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42)),
    ])

lr_pipeline = make_lr_pipeline(C=1.0)
gbt_pipeline = make_gbt_pipeline()

lr_pipeline.fit(X_train, y_train)
gbt_pipeline.fit(X_train, y_train)

# Assignment 4 helpers reused

def psi(expected, actual, bins=10):
    breaks = np.quantile(expected, np.linspace(0, 1, bins + 1))
    breaks[0], breaks[-1] = -np.inf, np.inf
    eps = 1e-6
    e = np.histogram(expected, breaks)[0] / len(expected) + eps
    a = np.histogram(actual, breaks)[0] / len(actual) + eps
    return float(np.sum((e - a) * np.log(e / a)))


def flag_psi(v):
    if v < 0.10:
        return "stable"
    if v < 0.25:
        return "monitor"
    return "RETRAIN"


def _error_rates_by_group(y_true, y_pred, groups):
    rows = []
    for g in sorted(pd.Series(groups).dropna().unique()):
        mask = (groups == g)
        y_g = y_true[mask]
        p_g = y_pred[mask]
        tn, fp, fn, tp = confusion_matrix(y_g, p_g, labels=[0, 1]).ravel()
        fpr = fp / (fp + tn) if (fp + tn) else np.nan
        fnr = fn / (fn + tp) if (fn + tp) else np.nan
        ppr = (tp + fp) / len(y_g)
        rows.append({"race": g, "n": int(len(y_g)), "FPR": fpr, "FNR": fnr, "PPR": ppr})
    return pd.DataFrame(rows)


def air(pipe, X_in, protected_race="African-American", reference_race="Caucasian"):
    preds = pipe.predict(X_in)
    d = X_in.copy()
    d["pred"] = preds
    p_protected = d.loc[d["race"] == protected_race, "pred"].mean()
    p_reference = d.loc[d["race"] == reference_race, "pred"].mean()
    return float(p_protected / p_reference) if p_reference > 0 else np.nan


def fpr_by_race(pipe, X_in, y_in):
    p = pipe.predict(X_in)
    out = _error_rates_by_group(y_in.values, p, X_in["race"].values)
    return {r["race"]: float(r["FPR"]) for _, r in out.iterrows()}


def get_numeric_scaler(pipe):
    return pipe.named_steps["preprocessor"].named_transformers_["num"]


def encode_numeric(pipe, X_in):
    scaler = get_numeric_scaler(pipe)
    return scaler.transform(X_in[numeric_features])


def decode_numeric(pipe, X_num_scaled):
    scaler = get_numeric_scaler(pipe)
    return scaler.inverse_transform(X_num_scaled)


def replace_numeric(X_template, X_num_scaled, pipe):
    X_new = X_template.copy()
    X_new.loc[:, numeric_features] = decode_numeric(pipe, X_num_scaled)
    return X_new

# Clean baseline table
baseline_rows = []
for model_name, model in [("LR", lr_pipeline), ("GBT", gbt_pipeline)]:
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    race_tbl = _error_rates_by_group(y_test.values, preds, X_test["race"].values)
    race_tbl["model"] = model_name
    race_tbl["AIR_AA_vs_Caucasian"] = air(model, X_test)
    race_tbl["AUC_test"] = roc_auc_score(y_test, probs)
    baseline_rows.append(race_tbl)

baseline_df = pd.concat(baseline_rows, ignore_index=True)
print(f"Filtered cohort size: {len(df)}")
print(f"Train/Test: {X_train.shape[0]}/{X_test.shape[0]}")
print("\nClean-model baseline by race:")
print(baseline_df[["model", "race", "n", "FPR", "FNR", "PPR", "AIR_AA_vs_Caucasian", "AUC_test"]].round(4).to_string(index=False))

## Part A — PGD Evasion Audit

We run PGD-style evasion attacks at `epsilon in {0.25, 0.5, 1.0, 2.0}` for both models.

- **LR direct attack:** analytic gradient (white-box)
- **GBT direct attack:** finite-difference gradient approximation (score-based)
- **Transfer attacks:** LR-crafted examples tested on GBT, and GBT-crafted examples tested on LR

For each `(model, epsilon)`, we report race-wise FPR and AIR (African-American / Caucasian).

In [ ]:
def _bce_loss(y_true, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))


def pgd_lr(pipe, X_base, y_true, epsilon, alpha=None, n_steps=20):
    if alpha is None:
        alpha = epsilon / 5 if epsilon > 0 else 0.0

    X0 = encode_numeric(pipe, X_base)
    X_adv = X0.copy()

    lr = pipe.named_steps["classifier"]
    pre = pipe.named_steps["preprocessor"]
    cat_encoder = pre.named_transformers_["cat"]
    X_cat = cat_encoder.transform(X_base[category_features])
    if hasattr(X_cat, "toarray"):
        X_cat = X_cat.toarray()

    n_num = len(numeric_features)
    w = lr.coef_.ravel()
    w_num, w_cat = w[:n_num], w[n_num:]
    b = float(lr.intercept_[0])
    cat_term = X_cat.dot(w_cat)

    for _ in range(n_steps):
        z = X_adv.dot(w_num) + cat_term + b
        p = expit(z)
        grad = (p - y_true.values)[:, None] * w_num[None, :]
        X_adv = X_adv + alpha * np.sign(grad)
        X_adv = np.clip(X_adv, X0 - epsilon, X0 + epsilon)

    return replace_numeric(X_base, X_adv, pipe)


def pgd_fd(pipe, X_base, y_true, epsilon, alpha=None, n_steps=8, h=1e-2):
    # Finite-difference PGD for non-differentiable tree model
    if alpha is None:
        alpha = epsilon / 4 if epsilon > 0 else 0.0

    X0 = encode_numeric(pipe, X_base)
    X_adv = X0.copy()
    n_feat = X_adv.shape[1]

    for _ in range(n_steps):
        grad = np.zeros_like(X_adv)
        for j in range(n_feat):
            Xp = X_adv.copy(); Xp[:, j] += h
            Xm = X_adv.copy(); Xm[:, j] -= h

            p_plus = pipe.predict_proba(replace_numeric(X_base, Xp, pipe))[:, 1]
            p_minus = pipe.predict_proba(replace_numeric(X_base, Xm, pipe))[:, 1]

            l_plus = _bce_loss(y_true.values, p_plus)
            l_minus = _bce_loss(y_true.values, p_minus)
            grad[:, j] = (l_plus - l_minus) / (2 * h)

        X_adv = X_adv + alpha * np.sign(grad)
        X_adv = np.clip(X_adv, X0 - epsilon, X0 + epsilon)

    return replace_numeric(X_base, X_adv, pipe)


def attack_metrics(pipe, X_attack, y_true, model_label, epsilon, attack_label):
    pred = pipe.predict(X_attack)
    acc = accuracy_score(y_true, pred)
    fprs = fpr_by_race(pipe, X_attack, y_true)
    out = {
        "attack": attack_label,
        "model_eval": model_label,
        "epsilon": epsilon,
        "accuracy": acc,
        "AIR_AA_vs_C": air(pipe, X_attack),
        "FPR_African-American": fprs.get("African-American", np.nan),
        "FPR_Caucasian": fprs.get("Caucasian", np.nan),
    }
    return out


eps_grid = [0.0, 0.25, 0.5, 1.0, 2.0]
rows_direct = []
rows_transfer = []

for eps in eps_grid:
    X_lr_adv = pgd_lr(lr_pipeline, X_test, y_test, epsilon=eps)
    X_gbt_adv = pgd_fd(gbt_pipeline, X_test, y_test, epsilon=eps)

    rows_direct.append(attack_metrics(lr_pipeline, X_lr_adv, y_test, "LR", eps, "direct"))
    rows_direct.append(attack_metrics(gbt_pipeline, X_gbt_adv, y_test, "GBT", eps, "direct"))

    # Transfer attacks
    rows_transfer.append(attack_metrics(gbt_pipeline, X_lr_adv, y_test, "GBT", eps, "transfer_LR_to_GBT"))
    rows_transfer.append(attack_metrics(lr_pipeline, X_gbt_adv, y_test, "LR", eps, "transfer_GBT_to_LR"))

part_a_direct = pd.DataFrame(rows_direct)
part_a_transfer = pd.DataFrame(rows_transfer)

print("Direct attack results (required table):")
print(part_a_direct.round(4).to_string(index=False))

# epsilon at which AIR first crosses below 0.80
cross_rows = []
for m in ["LR", "GBT"]:
    s = part_a_direct[part_a_direct["model_eval"] == m].sort_values("epsilon")
    crossed = s[s["AIR_AA_vs_C"] < 0.80]
    eps_cross = crossed["epsilon"].iloc[0] if len(crossed) else np.nan
    cross_rows.append({"model": m, "epsilon_crosses_0.80": eps_cross})

cross_df = pd.DataFrame(cross_rows)
print("\nAIR threshold crossing:")
print(cross_df.to_string(index=False))

# Plot AIR vs epsilon for direct and transfer
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, df_plot, title in [
    (axes[0], part_a_direct, "Direct Attack AIR vs Epsilon"),
    (axes[1], part_a_transfer, "Transfer Attack AIR vs Epsilon"),
]:
    for lab, grp in df_plot.groupby("model_eval"):
        ax.plot(grp["epsilon"], grp["AIR_AA_vs_C"], marker="o", label=lab)
    ax.axhline(0.80, color="red", linestyle="--", linewidth=1, label="AIR lower bound (0.80)")
    ax.axhline(1.25, color="gray", linestyle="--", linewidth=1, label="AIR upper bound (1.25)")
    ax.set_xlabel("epsilon")
    ax.set_title(title)
    ax.grid(alpha=0.25)

axes[0].set_ylabel("AIR (African-American / Caucasian)")
handles, labels = axes[0].get_legend_handles_labels()
uniq = dict(zip(labels, handles))
axes[1].legend(uniq.values(), uniq.keys(), fontsize=8)
plt.tight_layout()
plt.show()

### Part A Interpretation

**What the metric says:**
- AIR and race-specific FPR under adversarial perturbations quantify disparate impact under evasion pressure.
- The `epsilon_crosses_0.80` point identifies when impact-ratio fairness leaves the 80% rule region.

**What it misses:**
- PGD stress here perturbs only numeric features and assumes attacker control over those fields; real attackers may have different constraints.
- AIR/FPR alone do not capture calibration harms or downstream decision-policy effects.

**What action it justifies:**
- Prefer the model with slower AIR degradation under epsilon increase for high-stakes settings.
- Add proactive defenses (adversarial training, input checks) and reactive monitoring on race-conditioned error rates under suspicious query patterns.

## Part B — Poisoning Loop with Fairness Monitoring

We simulate training-time label-flip poisoning for two attacker variants:
- Target **African-American** rows
- Target **Caucasian** rows

For each flip rate, we retrain LR and track:
- Test AUC (performance degradation)
- AIR (fairness degradation)

Then we identify the **stealth zone** where `AUC drop <= 2pp` but `AIR` is outside `[0.80, 1.25]`.

In [ ]:
def poison_labels(X_in, y_in, target_race, flip_rate, seed=42):
    rng = np.random.default_rng(seed)
    y_poison = y_in.copy().astype(int)
    mask = (X_in["race"].values == target_race)
    idx = np.where(mask)[0]
    n_flip = int(np.floor(len(idx) * flip_rate))
    if n_flip > 0:
        flip_idx = rng.choice(idx, size=n_flip, replace=False)
        y_poison.iloc[flip_idx] = 1 - y_poison.iloc[flip_idx]
    return y_poison


poison_rates = [0.00, 0.05, 0.10, 0.15, 0.20, 0.30]
target_races = ["African-American", "Caucasian"]

clean_auc = roc_auc_score(y_test, lr_pipeline.predict_proba(X_test)[:, 1])
rows = []

for tr in target_races:
    for pr in poison_rates:
        y_p = poison_labels(X_train, y_train, tr, pr, seed=42)
        model = make_lr_pipeline(C=1.0)
        model.fit(X_train, y_p)

        prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, prob)
        air_val = air(model, X_test)

        rows.append({
            "target_race": tr,
            "poison_rate": pr,
            "AUC_test": auc,
            "AUC_drop_pp": clean_auc - auc,
            "AIR_AA_vs_C": air_val,
        })

part_b_df = pd.DataFrame(rows)
print("Poisoning loop results:")
print(part_b_df.round(4).to_string(index=False))

# Stealth zone
stealth = part_b_df[
    (part_b_df["AUC_drop_pp"] <= 0.02) &
    ((part_b_df["AIR_AA_vs_C"] < 0.80) | (part_b_df["AIR_AA_vs_C"] > 1.25))
].copy()

print("\nStealth zone (AUC drop <= 2pp and AIR outside [0.80, 1.25]):")
if len(stealth) == 0:
    print("No stealth-zone points found on this rate grid.")
else:
    print(stealth.round(4).to_string(index=False))

# Plot AUC and AIR together
fig, ax1 = plt.subplots(figsize=(8, 4.5))
for tr in target_races:
    g = part_b_df[part_b_df["target_race"] == tr]
    ax1.plot(g["poison_rate"], g["AUC_test"], marker="o", label=f"AUC ({tr})")
ax1.set_xlabel("Poison rate")
ax1.set_ylabel("AUC")
ax1.grid(alpha=0.25)

ax2 = ax1.twinx()
for tr in target_races:
    g = part_b_df[part_b_df["target_race"] == tr]
    ax2.plot(g["poison_rate"], g["AIR_AA_vs_C"], marker="s", linestyle="--", label=f"AIR ({tr})")
ax2.set_ylabel("AIR")
ax2.axhline(0.80, color="red", linestyle="--", linewidth=1)
ax2.axhline(1.25, color="gray", linestyle="--", linewidth=1)

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, fontsize=8, loc="best")
plt.title("Poisoning: AUC and AIR degradation by target-race variant")
plt.tight_layout()
plt.show()

# PSI monitor check: feature drift between clean train and poisoned train
psi_rows = []
for tr in target_races:
    for pr in poison_rates:
        # labels change only, so feature matrix is unchanged; this demonstrates monitor blindness
        for f in numeric_features:
            psi_val = psi(X_train[f].values, X_train[f].values)
            psi_rows.append({
                "target_race": tr,
                "poison_rate": pr,
                "feature": f,
                "PSI": psi_val,
                "flag": flag_psi(psi_val),
            })

part_b_psi = pd.DataFrame(psi_rows)
psi_summary = part_b_psi.groupby(["target_race", "poison_rate"], as_index=False)["PSI"].max()
print("\nPSI monitor check (max PSI across numeric features):")
print(psi_summary.round(6).to_string(index=False))
print("\nInterpretation cue: low PSI does not imply no poisoning risk when labels are attacked.")

### Part B Interpretation

**What the metric says:**
- AUC vs poison-rate and AIR vs poison-rate curves show how quickly model utility and group disparity degrade under targeted label manipulation.
- The stealth zone identifies realistic attack ranges where fairness can fail while top-line AUC still looks stable.

**What it misses:**
- Feature PSI stays near zero here because poisoning changes labels, not covariates; PSI cannot detect this attack class.
- Aggregate AUC can hide slice-level harm concentration.

**What action it justifies:**
- Add fairness-monitoring controls (AIR/FPR by race) to training-time governance, not just feature-drift checks.
- Add poisoning-specific controls (data provenance, RONI-style negative-impact filters, audit logs for relabeling events).

## Part C — Membership Inference Depth

We implement a shadow-model MI attack (Shokri et al. style):
1. Train shadow models and collect confidence outputs for members/non-members.
2. Train a meta-classifier to infer membership from confidence vectors.
3. Evaluate MI AUC on the target model's train/test records.

We run this for both LR and GBT, then test whether generalization gap predicts MI risk.
Finally, we sweep LR regularization strength `C in {0.01, 0.1, 1.0, 10.0}` and re-run MI.

In [ ]:
def build_attack_features(probs_pos):
    # Binary model outputs represented as [p0, p1] for meta-classifier
    p1 = np.clip(probs_pos, 1e-6, 1 - 1e-6)
    p0 = 1 - p1
    return np.column_stack([p0, p1])


def shadow_membership_attack(build_model_fn, X_train_t, y_train_t, X_test_t, y_test_t, n_shadow=5, seed=42):
    rng = np.random.default_rng(seed)

    pool_X = pd.concat([X_train_t, X_test_t], axis=0).reset_index(drop=True)
    pool_y = pd.concat([y_train_t, y_test_t], axis=0).reset_index(drop=True)

    meta_X, meta_y = [], []

    for i in range(n_shadow):
        # Random split to generate member/non-member examples for attack model training
        idx = np.arange(len(pool_X))
        rng.shuffle(idx)
        cut = int(0.5 * len(idx))
        in_idx, out_idx = idx[:cut], idx[cut:]

        X_in, y_in = pool_X.iloc[in_idx], pool_y.iloc[in_idx]
        X_out, y_out = pool_X.iloc[out_idx], pool_y.iloc[out_idx]

        shadow_model = build_model_fn()
        shadow_model.fit(X_in, y_in)

        p_in = shadow_model.predict_proba(X_in)[:, 1]
        p_out = shadow_model.predict_proba(X_out)[:, 1]

        meta_X.append(build_attack_features(p_in))
        meta_y.append(np.ones(len(p_in), dtype=int))
        meta_X.append(build_attack_features(p_out))
        meta_y.append(np.zeros(len(p_out), dtype=int))

    meta_X = np.vstack(meta_X)
    meta_y = np.concatenate(meta_y)

    attack_model = LogisticRegression(max_iter=1000, random_state=42)
    attack_model.fit(meta_X, meta_y)

    # Target-model membership inference
    target_model = build_model_fn()
    target_model.fit(X_train_t, y_train_t)

    p_mem = target_model.predict_proba(X_train_t)[:, 1]
    p_non = target_model.predict_proba(X_test_t)[:, 1]

    X_attack_eval = np.vstack([build_attack_features(p_mem), build_attack_features(p_non)])
    y_attack_eval = np.concatenate([np.ones(len(p_mem), dtype=int), np.zeros(len(p_non), dtype=int)])

    mi_scores = attack_model.predict_proba(X_attack_eval)[:, 1]
    mi_auc = roc_auc_score(y_attack_eval, mi_scores)

    train_auc = roc_auc_score(y_train_t, target_model.predict_proba(X_train_t)[:, 1])
    test_auc = roc_auc_score(y_test_t, target_model.predict_proba(X_test_t)[:, 1])

    confidence_member = np.maximum(1 - p_mem, p_mem)
    confidence_nonmember = np.maximum(1 - p_non, p_non)

    return {
        "mi_auc": float(mi_auc),
        "train_auc": float(train_auc),
        "test_auc": float(test_auc),
        "gen_gap": float(train_auc - test_auc),
        "conf_member": confidence_member,
        "conf_nonmember": confidence_nonmember,
    }


# MI for LR and GBT
mi_lr = shadow_membership_attack(lambda: make_lr_pipeline(C=1.0), X_train, y_train, X_test, y_test, n_shadow=5, seed=42)
mi_gbt = shadow_membership_attack(make_gbt_pipeline, X_train, y_train, X_test, y_test, n_shadow=5, seed=42)

mi_table = pd.DataFrame([
    {"model": "LR", "MI_AUC": mi_lr["mi_auc"], "train_AUC": mi_lr["train_auc"], "test_AUC": mi_lr["test_auc"], "gen_gap": mi_lr["gen_gap"]},
    {"model": "GBT", "MI_AUC": mi_gbt["mi_auc"], "train_AUC": mi_gbt["train_auc"], "test_AUC": mi_gbt["test_auc"], "gen_gap": mi_gbt["gen_gap"]},
])

print("MI and generalization summary:")
print(mi_table.round(4).to_string(index=False))

# Confidence-gap histograms side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, obj, title in [
    (axes[0], mi_lr, "LR"),
    (axes[1], mi_gbt, "GBT"),
]:
    ax.hist(obj["conf_member"], bins=30, alpha=0.55, label="member")
    ax.hist(obj["conf_nonmember"], bins=30, alpha=0.55, label="non-member")
    ax.set_title(f"Confidence Gap: {title}")
    ax.set_xlabel("max-class confidence")
    ax.grid(alpha=0.2)
axes[0].set_ylabel("count")
axes[1].legend()
plt.tight_layout()
plt.show()

# LR C-sweep: MI AUC vs utility
c_grid = [0.01, 0.1, 1.0, 10.0]
rows_c = []
for c in c_grid:
    mi_c = shadow_membership_attack(lambda cval=c: make_lr_pipeline(C=cval), X_train, y_train, X_test, y_test, n_shadow=4, seed=42)
    rows_c.append({
        "C": c,
        "MI_AUC": mi_c["mi_auc"],
        "test_AUC": mi_c["test_auc"],
        "gen_gap": mi_c["gen_gap"],
    })

c_df = pd.DataFrame(rows_c).sort_values("C")
print("\nLR regularization sweep:")
print(c_df.round(4).to_string(index=False))

plt.figure(figsize=(7, 4))
plt.plot(np.log10(c_df["C"]), c_df["MI_AUC"], marker="o", label="MI AUC")
plt.plot(np.log10(c_df["C"]), c_df["test_AUC"], marker="s", label="Test AUC")
plt.xlabel("log10(C)")
plt.ylabel("Metric value")
plt.title("LR Regularization Tradeoff: Privacy vs Utility")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

### Part C Interpretation

**What the metric says:**
- MI AUC above 0.5 means an attacker can infer training membership better than chance.
- Confidence-gap histograms visualize the leakage signal (members often receive higher confidence).
- The `gen_gap` vs `MI_AUC` comparison operationalizes the lecture claim that overfitting increases privacy risk.

**What it misses:**
- MI AUC depends on attack strength assumptions (shadow model quality, attack classifier choice).
- Two-model correlation checks (LR vs GBT) are suggestive, not statistically definitive.

**What action it justifies:**
- Reduce overfitting (regularization, early stopping, simpler models) as a privacy control.
- Use periodic red-team MI testing in model governance.
- For high-sensitivity settings, consider stronger privacy controls (for example DP training) and explicitly document minority-accuracy tradeoffs.

## Part D — Reflection (Highest-Risk Finding and Mitigations)

### Highest-risk finding
The most consequential risk in this notebook is **targeted label-flip poisoning** because it can move fairness metrics outside policy bounds while headline model quality remains nearly unchanged (the stealth zone). This is dangerous in governance workflows that rely on aggregate AUC as the primary quality gate.

### One proactive mitigation (with measurable effect)
A proactive control is a **data-ingestion integrity gate with RONI-style filtering** before model retraining. In this notebook's framing, the measurable target is to reduce stealth-zone width (fewer poison rates where `AUC_drop <= 0.02` and `AIR outside [0.80, 1.25]`).

### One reactive mitigation (with measurable effect)
A reactive control is a **post-training fairness and privacy monitor**: periodic AIR-by-race plus membership-inference red-team checks. From Part C, stronger LR regularization (`C` down) should reduce MI AUC; that can be used as a bounded response when privacy risk rises.

### Disparate-impact note on mitigations
Mitigations can introduce their own tradeoffs. Stronger regularization often lowers utility and can shift subgroup error rates; therefore each mitigation should be audited for race-specific FPR/FNR changes before adoption.

### Tie-back to Lecture 05 core principle
Lecture 05 emphasizes that brittleness under ordinary shift and vulnerability under adversarial pressure share a common root: over-reliance on non-robust features. This notebook's results are consistent with that principle across evasion, poisoning, and membership inference.

## Verification Checks

Run this section after all prior cells.

Expected checks:
1. `pgd_lr(..., epsilon=0)` should be a no-op (predictions unchanged).
2. `poison_labels(..., flip_rate=0)` should return identical labels.
3. MI AUC on random labels should be near chance (around 0.5).

In [ ]:
# 1) Zero-epsilon PGD no-op
X_lr_eps0 = pgd_lr(lr_pipeline, X_test, y_test, epsilon=0.0)
pred_clean = lr_pipeline.predict(X_test)
pred_eps0 = lr_pipeline.predict(X_lr_eps0)
no_op_match = float((pred_clean == pred_eps0).mean())

# 2) Zero-rate poisoning no-op
y_poison0 = poison_labels(X_train, y_train, target_race="African-American", flip_rate=0.0, seed=42)
poison_no_op = float((y_poison0.values == y_train.values).mean())

# 3) MI AUC on random labels (approx chance)
rng = np.random.default_rng(42)
y_train_rand = pd.Series(rng.integers(0, 2, size=len(y_train)), index=y_train.index)
y_test_rand = pd.Series(rng.integers(0, 2, size=len(y_test)), index=y_test.index)
mi_rand = shadow_membership_attack(lambda: make_lr_pipeline(C=1.0), X_train, y_train_rand, X_test, y_test_rand, n_shadow=3, seed=42)

print("Verification summary:")
print(f"  PGD epsilon=0 prediction match rate: {no_op_match:.4f} (expected 1.0)")
print(f"  Poison flip_rate=0 label match rate:  {poison_no_op:.4f} (expected 1.0)")
print(f"  Random-label MI AUC:                  {mi_rand['mi_auc']:.4f} (expected near 0.5)")